# CLV 거래활동·거래가치 이중축 임베딩 1단계 screening

Dunnhumby와 H&M 60일을 순차 실행합니다. 각 데이터에서 seed 42, validation-only로 `M1`, `dual_clv_fixed`, `dual_shuffled_user`, `dual_adapter_only`만 실행하며 test와 holdout은 생성하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REVIEWED_SHA = 'cd1bb457a3e78b420539979d41c5ba8f5cc2ac70'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-dual')
assert not REPO_DIR.exists(), '새 Colab 런타임에서 실행하세요.'
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('검토된 소스:', actual_sha)


In [ ]:
import json, torch
from lightgcn_clv_dual import MODELS, configure_dual_run, preflight_summary, run_experiment

DATASET_PRESETS = ('dunnhumby', 'hm_w60')
RESULT_ROOT = Path('/content/drive/MyDrive/논문/data')
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
assert tuple(MODELS) == ('m1', 'dual_clv_fixed', 'dual_shuffled_user', 'dual_adapter_only')
configs = {
    'dunnhumby': configure_dual_run(
        'dunnhumby',
        out_dir=str(RESULT_ROOT / 'results_clv_dual_dunnhumby'),
        m1_checkpoint_dir=str(RESULT_ROOT / 'results_v3_dunnhumby'),
    ),
    'hm_w60': configure_dual_run(
        'hm', short_hm=True,
        out_dir=str(RESULT_ROOT / 'results_clv_dual_hm_w60'),
        m1_checkpoint_dir=str(RESULT_ROOT / 'results_v3_hm_w60'),
    ),
}
for dataset_preset in DATASET_PRESETS:
    print(f'--- {dataset_preset} 실행 설정 ---')
    print(json.dumps(preflight_summary(configs[dataset_preset]), ensure_ascii=False, indent=2))


In [ ]:
results_by_dataset = {}
run_errors = {}
for dataset_preset in DATASET_PRESETS:
    print(f'\n===== {dataset_preset} 실행 시작 =====')
    try:
        results_by_dataset[dataset_preset] = run_experiment(configs[dataset_preset])
        print(f'===== {dataset_preset} 저장 완료 =====')
    except Exception as exc:
        run_errors[dataset_preset] = repr(exc)
        print(f'===== {dataset_preset} 실패: {exc!r} =====')
        torch.cuda.empty_cache()
print('실행 완료:', list(results_by_dataset))
print('실패:', run_errors)


In [ ]:
from IPython.display import display
import pandas as pd

for dataset_preset, result_df in results_by_dataset.items():
    print(f'\n===== {dataset_preset} 결과 =====')
    display(result_df.sort_values(['model_id', 'gate_shape', 'lambda']))
    print('선택 운영점:', result_df.attrs['selected_operating_point'])
    print('최종 screening 판정:', result_df.attrs['screening_decision'])
    delta_path = Path(result_df.attrs['result_paths']['delta_csv'])
    print('M1 대비 paired delta:')
    display(pd.read_csv(delta_path).sort_values(['model_id', 'gate_shape', 'lambda', 'metric']))
    print('결과 파일:')
    for label, path in result_df.attrs['result_paths'].items():
        print(f' - {label}: {path}')
if run_errors:
    print('실패한 데이터:', run_errors)
